<a href="https://colab.research.google.com/github/sumitpaul398/ML-projects/blob/main/plate_with_holes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parametric model of plate with multiple holes

### Objective:
In this example we'll setup a parametric model in PyMAPDL for a rectangular plate with multiple holes. The model is setup such that one can change the dimensions of the plate, the number of holes and their radius, the material properties and the pressure applied.

The learning objectives of this demo are:
* Launch PyMAPDL on a local machine
* Setup and solve a parametric model using PyMAPDL
* Interactive plotting of CAD, mesh, and results in Pythonic interface.

### Model Details
<div>
    <img src="attachment:plate_with_hole.png" width="500"/>
</div>

#### Model parameters:
* Length, width and depth of the plate
* Number of holes
* Raidus of the holes
* Material properties (Young's modulus and Poisson's ratio)
* Applied pressure

## Step 1 - define all parameters

In [1]:
pip install ansys-mapdl-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 41.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 MB 53.0 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 46.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [ansys-mapdl-core]


In [2]:
pip install ansys-mapdl-core[graphics]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 830.4/830.4 kB 16.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 43.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [ansys-tools-visualization-interface]


In [3]:
# All units in (m, Kg, s)
LENGTH = 5
WIDTH = 2.5
DEPTH = 0.1
RADIUS = 0.5
NUM = 3

E = 2e11
NU = 0.27

PRESSURE = 1000

# Step 2 - launch MAPDL and create geometry

In [ ]:
from ansys.mapdl.core import launch_mapdl
mapdl = launch_mapdl()

mapdl.clear()
mapdl.prep7()
mapdl.block(0, LENGTH, 0, WIDTH, 0, DEPTH)
for i in range(1,NUM+1):
    mapdl.cyl4(i*LENGTH/(NUM+1),WIDTH/2,RADIUS,'','','',2*DEPTH)
mapdl.vsbv(1,'all')
mapdl.vplot('all')

Cached mapdl executable not found
You are about to enter manually the path of the Ansys MAPDL executable

(ansysxxx, where XXX is the version
This file is very likely to contained in path ending in 'vXXX/ansys/bin/ansysXXX'.


If you experience problems with the input path you can overwrite the configuration
file by typing:
>>> from ansys.tools.common.path import save_mapdl_path
>>> save_mapdl_path('/new/path/to/executable/')

Enter the location of Ansys MAPDL (ansysxxx):C:\Program Files\ANSYS Inc
The supplied path is either: not a valid file path, or does not match 'ansysxxx' name.


## Step 3 - define material properties, mesh attributes and generate mesh.

In [ ]:
mapdl.lesize("ALL", 0.15, layer1=1)

mapdl.mp('ex',1,E)
mapdl.mp('nuxy',1,NU)

mapdl.et(1,'SOLID186')
mapdl.mshape(1, "3D")
mapdl.mshkey(0)
mapdl.vmesh('all')
mapdl.eplot()

## Step 4 - apply loads and boundary conditions

In [ ]:
mapdl.nsel('s','loc','x',0)
mapdl.d('all','all',0)

mapdl.nsel('s','loc','x', LENGTH)
mapdl.sf('all','pres',PRESSURE)

mapdl.allsel()
mapdl.finish()

## Step 4 - solve the static problem

In [ ]:
mapdl.slashsolu()
mapdl.solve()
mapdl.finish()

In [ ]:
# enter the solver routine and solve
mapdl.slashsolu()
output = mapdl.solve()

print(output)

## Step 5 - plot the stress contours for the model

In [ ]:
result = mapdl.result
result.plot_principal_nodal_stress(0,'seqv',background='w',show_edges=True,text_color='k',add_text=True)

## Step 6 - exit MAPDL

In [ ]:
mapdl.exit()